# Lesson 5.4 — Alert Troubleshooting and Tuning

## Learning Objectives
- Diagnose common alerting problems (too many alerts, missed alerts)
- Tune health rule sensitivity using load guards, baselines, and grace periods
- Investigate health rule violations in the Controller UI
- Use the Alert Sensitivity Tuning guidance from AppDynamics docs

**Reference**: [Troubleshoot Health Rule Violations](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/troubleshoot-health-rule-violations) | [Alert Sensitivity Tuning](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/alert-sensitivity-tuning-questions-and-answers)

## 1. Common Alerting Problems

### Problem 1: Alert Flooding (Too Many Alerts)

**Symptoms**: Email inbox overwhelmed, Slack channel noisy, PagerDuty burnout.

**Common causes and fixes**:

| Cause | Fix |
|-------|-----|
| Health rule fires on every single BT | Reduce scope to critical BTs only |
| Low load guard — alerts on 1 call per minute | Increase load guard (e.g., min 50 calls/min) |
| Policy has no suppression period | Set minimum 15-30 minute re-alert period |
| Baseline hasn't learned yet | Wait 2 weeks or use static threshold temporarily |
| Too many "continues" events | Change policy to only fire on "starts" |

### Problem 2: Missed Alerts (False Negatives)

**Symptoms**: Users complain before any alert fires.

**Common causes and fixes**:

| Cause | Fix |
|-------|-----|
| Health rule scope excludes the problematic BT | Widen scope |
| Load guard too high (no alerts during low-traffic test) | Lower load guard |
| Alert suppression too long | Reduce suppression period |
| Baseline has absorbed the slowdown as "normal" | Reset baseline after fixing performance issues |

In [ ]:
# Diagnose alert sensitivity issues

def diagnose_alert_sensitivity(rule_config: dict, observations: list) -> list:
    """
    Analyze a health rule config and historical observations to find issues.
    Returns a list of diagnostic messages.
    """
    issues = []
    
    # Check 1: Load guard
    load_guard = rule_config.get('load_guard_cpm', 0)
    if load_guard == 0:
        issues.append("WARNING: No load guard set — rule can fire on 1 call per minute. Add a load guard of at least 10 cpm.")
    elif load_guard > 500:
        issues.append(f"WARNING: Very high load guard ({load_guard} cpm) — rule won't fire during low-traffic periods.")
    
    # Check 2: Baseline multiplier
    multiplier = rule_config.get('baseline_multiplier', 1.0)
    if multiplier < 2.0:
        issues.append(f"WARNING: Very sensitive baseline multiplier ({multiplier}×) — may cause frequent false alerts.")
    
    # Check 3: Policy suppression
    suppress_min = rule_config.get('suppress_minutes', 0)
    if suppress_min == 0:
        issues.append("WARNING: No alert suppression — will re-alert every minute during sustained incidents.")
    
    # Check 4: Alert rate from observations
    if observations:
        alert_rate = sum(1 for o in observations if o['violation']) / len(observations)
        if alert_rate > 0.3:
            issues.append(f"NOISE: Rule fires {alert_rate*100:.0f}% of the time — too sensitive. Consider adjusting thresholds.")
        elif alert_rate == 0:
            issues.append("SILENT: Rule never fires — check scope, thresholds, and that data is flowing.")
        else:
            issues.append(f"OK: Rule fires {alert_rate*100:.1f}% of the time — reasonable sensitivity.")
    
    return issues if issues else ["No issues detected — rule is well-configured."]

# Test with different configurations
test_configs = [
    {
        "name": "Too sensitive (no load guard, low multiplier, no suppression)",
        "config": {"load_guard_cpm": 0, "baseline_multiplier": 1.5, "suppress_minutes": 0},
        "observations": [{"violation": True} for _ in range(40)] + [{"violation": False} for _ in range(60)]
    },
    {
        "name": "Too insensitive (high load guard)",
        "config": {"load_guard_cpm": 1000, "baseline_multiplier": 4.0, "suppress_minutes": 60},
        "observations": [{"violation": False} for _ in range(100)]
    },
    {
        "name": "Well-tuned rule",
        "config": {"load_guard_cpm": 50, "baseline_multiplier": 3.0, "suppress_minutes": 30},
        "observations": [{"violation": True} for _ in range(5)] + [{"violation": False} for _ in range(95)]
    },
]

for tc in test_configs:
    print(f"\n=== Config: {tc['name']} ===")
    issues = diagnose_alert_sensitivity(tc['config'], tc['observations'])
    for issue in issues:
        print(f"  {'⚠️' if 'WARNING' in issue or 'NOISE' in issue or 'SILENT' in issue else '✅'}  {issue}")

## 2. Investigating a Health Rule Violation

When you receive an alert, here's how to investigate in the Controller UI:

### Step-by-step investigation:

1. **Find the violation**:
   - Navigate to **Alert & Respond → Health Rule Violations**
   - Or click the **red/yellow indicator** in the Flow Map
   - Filter by time range and severity

2. **Read the violation details**:
   - Which health rule fired?
   - Which BT or tier was affected?
   - What were the actual metric values vs. the threshold?
   - Duration: when did it start and end?

3. **Drill into the cause**:
   - Click the affected BT to open its performance view
   - Check **Transaction Snapshots** for the violation time window
   - Compare with the **Metric Browser** (response time during violation)
   - Check **Infrastructure** metrics for the affected tier (CPU? Memory? GC?)

4. **Correlate with events**:
   - Was there a **deployment** around the same time? (check Events tab)
   - Was there a **database slow query** spike? (Database Visibility)
   - Was there external traffic spike? (EUM data)

In [ ]:
# Simulate a health rule violation investigation workflow

investigation_checklist = {
    "alert": {
        "rule": "Checkout SLA Violation",
        "severity": "CRITICAL",
        "start_time": "2026-07-08 14:23:00 UTC",
        "affected_bt": "/checkout",
        "affected_tier": "OrderService",
        "metric_value": "2847ms",
        "threshold": "2000ms",
    },
    "investigation_steps": [
        {
            "step": 1,
            "action": "Check transaction snapshots for /checkout during 14:20-14:30",
            "ui_path": "Applications > StoreFront > Business Transactions > /checkout > Transaction Snapshots",
            "finding": "PENDING — TODO: Fill in what you found"
        },
        {
            "step": 2,
            "action": "Check OrderService node CPU and memory at 14:23",
            "ui_path": "Applications > StoreFront > Servers > OrderService > order-svc-1",
            "finding": "PENDING"
        },
        {
            "step": 3,
            "action": "Check Database Visibility for slow queries at 14:23",
            "ui_path": "Databases > StoreFront-MySQL > Query Monitor",
            "finding": "PENDING"
        },
        {
            "step": 4,
            "action": "Check Events for deployments or config changes",
            "ui_path": "Applications > StoreFront > Events",
            "finding": "PENDING"
        },
        {
            "step": 5,
            "action": "ADQL: Find all /checkout transactions > 2000ms at that time",
            "ui_path": "Analytics > Searches > New Search",
            "adql": "SELECT transactionName, responseTime, tierName FROM transactions WHERE transactionName = '/checkout' AND responseTime > 2000 ORDER BY responseTime DESC LIMIT 20",
            "finding": "PENDING"
        },
    ]
}

print("=== Health Rule Violation Investigation ===")
print(f"Rule    : {investigation_checklist['alert']['rule']}")
print(f"Severity: {investigation_checklist['alert']['severity']}")
print(f"Time    : {investigation_checklist['alert']['start_time']}")
print(f"BT      : {investigation_checklist['alert']['affected_bt']}")
print(f"Actual  : {investigation_checklist['alert']['metric_value']} (threshold: {investigation_checklist['alert']['threshold']})")
print()
print("Investigation Steps:")
for step in investigation_checklist['investigation_steps']:
    print(f"\n  Step {step['step']}: {step['action']}")
    print(f"    UI path: {step['ui_path']}")
    if 'adql' in step:
        print(f"    ADQL: {step['adql']}")
    print(f"    Finding: {step['finding']}")

In [ ]:
# EXERCISE 5.4: Alert sensitivity tuning
#
# You have a health rule that fires 45% of the time (way too noisy).
# Current config: load_guard=0, baseline_multiplier=1.5, suppress_minutes=0
#
# TASK: Recommend a tuned configuration that reduces alert rate to ~5%
# while still catching genuine incidents.

import random
random.seed(100)

# Simulate 1000 1-minute windows
# 95% of the time: normal traffic (baseline_rt=300, std=50)
# 5% of the time: genuine incident (rt is 5× baseline)
windows = []
for i in range(1000):
    is_incident = random.random() < 0.05
    rt = random.gauss(1500 if is_incident else 300, 50)
    cpm = random.gauss(80 if is_incident else 150, 20)
    windows.append({"rt": max(50, rt), "cpm": max(0, cpm), "is_incident": is_incident})

def evaluate_rule(windows, load_guard_cpm, rt_threshold_ms):
    """Count how many windows trigger the rule."""
    alerts = sum(1 for w in windows if w['rt'] > rt_threshold_ms and w['cpm'] >= load_guard_cpm)
    real_incidents = sum(1 for w in windows if w['is_incident'])
    caught = sum(1 for w in windows if w['is_incident'] and w['rt'] > rt_threshold_ms and w['cpm'] >= load_guard_cpm)
    return alerts, caught, real_incidents

# Current noisy config
current_alerts, current_caught, total_incidents = evaluate_rule(windows, 0, 450)  # 1.5× baseline
print("=== Current (noisy) config ===")
print(f"  Load guard: 0 cpm, Threshold: 450ms")
print(f"  Total alerts : {current_alerts}/1000 ({current_alerts/10:.1f}%)")
print(f"  True positives: {current_caught}/{total_incidents} incidents caught")
print(f"  False positives: {current_alerts - current_caught}")
print()

# TODO: Tune the config below to reduce false positives while keeping true positive rate high
tuned_load_guard = 0    # TODO: change this
tuned_threshold  = 450  # TODO: change this (hint: use 3× baseline = 900ms)

tuned_alerts, tuned_caught, _ = evaluate_rule(windows, tuned_load_guard, tuned_threshold)
print("=== Tuned config ===")
print(f"  Load guard: {tuned_load_guard} cpm, Threshold: {tuned_threshold}ms")
print(f"  Total alerts : {tuned_alerts}/1000 ({tuned_alerts/10:.1f}%)")
print(f"  True positives: {tuned_caught}/{total_incidents} incidents caught")
print(f"  False positives: {tuned_alerts - tuned_caught}")

## Summary

- **Too noisy**: Add load guards, increase baseline multiplier, add suppression periods
- **Too silent**: Check rule scope, lower load guards, verify data flow
- When a violation fires, investigate: Transaction Snapshots → Infrastructure → Database → Events timeline
- ADQL is a powerful tool for violation investigation — pinpoint which specific transactions were affected
- Reset baselines after fixing performance issues to prevent the slowdown becoming the "new normal"

---

**Phase 5 Complete!** Proceed to [Phase 6 — Grafana Integration](../phase-6-grafana-integration/lesson-6.1-grafana-datasource-setup.ipynb)